# Amazon ML Challenge 2026 — End-to-End Training Pipeline
## Business Entity Resolution

**Instructions:**
1. Upload your dataset to Google Drive
2. Update the `DATA_DIR` path in Cell 2
3. Run all cells in order
4. Download `output/matching_results.tsv` and upload to the portal

> Runtime: GPU T4 recommended (free on Colab)


In [ ]:
# ─── CELL 1: Mount Google Drive + Install packages ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q lightgbm scikit-learn pandas numpy rapidfuzz jellyfish phonetics datasketch pyarrow scipy

In [ ]:
# ─── CELL 2: CONFIGURE PATHS (edit these) ────────────────────────────────────
import os
from pathlib import Path

# EDIT THIS: where you uploaded the student_resource folder in Google Drive
DATA_DIR = Path('/content/drive/MyDrive/amazon_dataset/6ab10eb3b23ba_student_resource/student_resource/dataset')

# Working dirs (created automatically)
WORK_DIR    = Path('/content/amazon_ml_solution')
PROCESSED   = WORK_DIR / 'data' / 'processed'
MODELS_DIR  = WORK_DIR / 'models'
EXPERIMENTS = WORK_DIR / 'experiments'
OUTPUT_DIR  = WORK_DIR / 'output'

for d in [PROCESSED, MODELS_DIR, EXPERIMENTS, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'
print('Data exists:', DATA_DIR.exists())
print('Files:', list(TRAIN_DIR.iterdir()))

In [ ]:
# ─── CELL 3: Clone solution code ─────────────────────────────────────────────
# Option A: Clone from your repo
# !git clone https://github.com/YOUR_USERNAME/amazon_ml_solution.git /content/amazon_ml_solution

# Option B: Upload the src/ folder manually and set path
import sys
sys.path.insert(0, '/content/amazon_ml_solution/src')  # adjust if needed
print('Python path set.')

In [ ]:
# ─── CELL 4: EDA — understand data scale ─────────────────────────────────────
import pandas as pd
import numpy as np
import json
import logging
logging.basicConfig(level=logging.INFO)

print('Loading source files ...')
s1 = pd.read_csv(TRAIN_DIR / 'train_source1.tsv', sep='\t', dtype=str).fillna('')
s2 = pd.read_csv(TRAIN_DIR / 'train_source2.tsv', sep='\t', dtype=str).fillna('')
s3 = pd.read_csv(TRAIN_DIR / 'train_source3.tsv', sep='\t', dtype=str).fillna('')
gt = pd.read_csv(TRAIN_DIR / 'train_ground_truth.tsv', sep='\t', dtype=str).fillna('')

print(f'S1: {len(s1):,}  S2: {len(s2):,}  S3: {len(s3):,}  GT: {len(gt):,}')
print('Countries in S1:', s1['country'].value_counts().to_dict())
print('Countries in S2:', s2['country'].value_counts().to_dict())

# Ground truth analysis
gt_dict = {}
for _, row in gt.iterrows():
    ids = set(row['matched_entity_ids'].split(',')) if row['matched_entity_ids'].strip() else set()
    gt_dict[row['source1_entity_id']] = ids

match_counts = [len(v) for v in gt_dict.values()]
n_singletons = sum(1 for v in match_counts if v == 0)
print(f'\nSingleton rate: {n_singletons/len(match_counts):.1%}')
print(f'Max matches per S1: {max(match_counts)}')
print(f'Avg matches (non-zero): {np.mean([v for v in match_counts if v>0]):.2f}')

In [ ]:
# ─── CELL 5: Preprocessing ───────────────────────────────────────────────────
import re
import unicodedata

NAME_ABBREV = {
    'corp': 'corporation', 'corp.': 'corporation', 'pvt': 'private', 'pvt.': 'private',
    'ltd': 'limited', 'ltd.': 'limited', 'inc': 'incorporated', 'inc.': 'incorporated',
    'co': 'company', 'co.': 'company', 'llc': 'limited liability company',
    'llp': 'limited liability partnership', 'intl': 'international', 'mfg': 'manufacturing',
    'svcs': 'services', 'assoc': 'associates', 'bros': 'brothers', 'grp': 'group',
    'mgmt': 'management', 'tech': 'technology', 'sys': 'systems',
}
ADDR_ABBREV = {
    'rd': 'road', 'rd.': 'road', 'st': 'street', 'ave': 'avenue', 'ave.': 'avenue',
    'blvd': 'boulevard', 'dr': 'drive', 'ln': 'lane', 'hwy': 'highway',
    'fl': 'floor', 'apt': 'apartment', 'ste': 'suite', 'bldg': 'building',
}
PUNCT_RE  = re.compile(r'[^\w\s\-]', re.UNICODE)
SPACE_RE  = re.compile(r'\s+')
AMP_RE    = re.compile(r'\s*&\s*', re.UNICODE)
POSTAL_RE = re.compile(r'\b(\d{5,6})\b')
NUMS_RE   = re.compile(r'\b\d+\b')

def normalize_name(text):
    if not text or not text.strip(): return ''
    s = unicodedata.normalize('NFC', text)
    s = AMP_RE.sub(' and ', s).lower()
    s = PUNCT_RE.sub(' ', s)
    s = SPACE_RE.sub(' ', s).strip()
    tokens = [NAME_ABBREV.get(t, t) for t in s.split()]
    return ' '.join(tokens)

def normalize_address(text):
    if not text or not text.strip(): return ''
    s = unicodedata.normalize('NFC', text)
    s = AMP_RE.sub(' and ', s).lower()
    s = PUNCT_RE.sub(' ', s)
    s = SPACE_RE.sub(' ', s).strip()
    tokens = [ADDR_ABBREV.get(t, t) for t in s.split()]
    return ' '.join(tokens)

def extract_postal(text):
    m = POSTAL_RE.search(str(text))
    return m.group(1) if m else ''

def sorted_tokens(text):
    return ' '.join(sorted(text.split()))

def preprocess_df(df):
    df = df.copy()
    df['name_clean']   = df['business_name'].apply(normalize_name)
    df['name_sorted']  = df['name_clean'].apply(sorted_tokens)
    df['name_prefix4'] = df['name_sorted'].str[:4]
    df['addr_clean']   = df['business_address'].apply(normalize_address)
    df['postal_code']  = df['business_address'].apply(extract_postal)
    df['country_norm'] = df['country'].str.lower().str.strip()
    df['embed_text']   = df['name_clean'] + ' [SEP] ' + df['addr_clean']
    return df

print('Preprocessing ...')
s1p = preprocess_df(s1);  print('S1 done')
s2p = preprocess_df(s2);  print('S2 done')
s3p = preprocess_df(s3);  print('S3 done')

# Save to parquet
s1p.to_parquet(PROCESSED / 'train_s1.parquet', index=False)
s2p.to_parquet(PROCESSED / 'train_s2.parquet', index=False)
s3p.to_parquet(PROCESSED / 'train_s3.parquet', index=False)
print('Parquets saved.')

In [ ]:
# ─── CELL 6: BLOCKING — TF-IDF + Postal + Phonetic ───────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pickle
from collections import defaultdict

pool_df = pd.concat([s2p, s3p], ignore_index=True)
pool_ids = pool_df['entity_id'].tolist()
s1_ids   = s1p['entity_id'].tolist()

print(f'Blocking: {len(s1p):,} S1 vs {len(pool_df):,} pool records')

candidates = defaultdict(set)
TOP_K = 50
BATCH = 5000

# --- Pass A: Exact name match ---
name_idx = defaultdict(list)
for _, row in pool_df.iterrows():
    if row['name_clean']:
        name_idx[row['name_clean']].append(row['entity_id'])
for _, row in s1p.iterrows():
    if row['name_clean'] in name_idx:
        candidates[row['entity_id']].update(name_idx[row['name_clean']])
print(f'Pass A done. Pairs so far: {sum(len(v) for v in candidates.values()):,}')

# --- Pass B: Char TF-IDF ---
print('Building char TF-IDF index ...')
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), sublinear_tf=True, min_df=1)
idx_mat  = char_vec.fit_transform(pool_df['name_clean'].fillna('').tolist())
pool_arr = np.array(pool_ids)

for start in range(0, len(s1p), BATCH):
    end    = min(start + BATCH, len(s1p))
    batch  = s1p.iloc[start:end]
    q_mat  = char_vec.transform(batch['name_clean'].fillna('').tolist())
    sims   = (q_mat @ idx_mat.T).toarray()
    for i, (_, row) in enumerate(batch.iterrows()):
        top_idx = np.argpartition(sims[i], -min(TOP_K, len(sims[i])))[-TOP_K:]
        top_idx = top_idx[np.argsort(sims[i][top_idx])[::-1]]
        good    = top_idx[sims[i][top_idx] > 0]
        candidates[row['entity_id']].update(pool_arr[good].tolist())
    if (start // BATCH) % 50 == 0:
        print(f'  char TF-IDF: {end:,}/{len(s1p):,}')
print(f'Pass B done. Pairs: {sum(len(v) for v in candidates.values()):,}')

# --- Pass C: Token TF-IDF ---
print('Building token TF-IDF index ...')
tok_vec  = TfidfVectorizer(analyzer='word', ngram_range=(1,2), sublinear_tf=True, min_df=1)
idx_mat2 = tok_vec.fit_transform(pool_df['name_clean'].fillna('').tolist())
for start in range(0, len(s1p), BATCH):
    end   = min(start + BATCH, len(s1p))
    batch = s1p.iloc[start:end]
    q_mat = tok_vec.transform(batch['name_clean'].fillna('').tolist())
    sims  = (q_mat @ idx_mat2.T).toarray()
    for i, (_, row) in enumerate(batch.iterrows()):
        top_idx = np.argpartition(sims[i], -min(TOP_K, len(sims[i])))[-TOP_K:]
        top_idx = top_idx[np.argsort(sims[i][top_idx])[::-1]]
        good    = top_idx[sims[i][top_idx] > 0]
        candidates[row['entity_id']].update(pool_arr[good].tolist())
    if (start // BATCH) % 50 == 0:
        print(f'  token TF-IDF: {end:,}/{len(s1p):,}')
print(f'Pass C done. Pairs: {sum(len(v) for v in candidates.values()):,}')

# --- Pass D: Postal code blocking ---
postal_idx = defaultdict(list)
for _, row in pool_df.iterrows():
    pc = row['postal_code'].strip()
    if pc:
        key = f"{row['country_norm']}|{pc}"
        postal_idx[key].append(row['entity_id'])
for _, row in s1p.iterrows():
    pc = row['postal_code'].strip()
    if pc:
        key = f"{row['country_norm']}|{pc}"
        if key in postal_idx and len(postal_idx[key]) <= 500:
            candidates[row['entity_id']].update(postal_idx[key])
print(f'Pass D done. Pairs: {sum(len(v) for v in candidates.values()):,}')

# --- Pass G: Sorted prefix blocking ---
prefix_idx = defaultdict(list)
for _, row in pool_df.iterrows():
    pfx = row['name_prefix4'][:4]
    if pfx:
        key = f"{row['country_norm']}|{pfx}"
        prefix_idx[key].append(row['entity_id'])
for _, row in s1p.iterrows():
    pfx = row['name_prefix4'][:4]
    if pfx:
        key = f"{row['country_norm']}|{pfx}"
        if key in prefix_idx and len(prefix_idx[key]) <= 200:
            candidates[row['entity_id']].update(prefix_idx[key])
print(f'Pass G done. Pairs: {sum(len(v) for v in candidates.values()):,}')

# Cap per S1
for s1_id in list(candidates.keys()):
    if len(candidates[s1_id]) > 500:
        candidates[s1_id] = set(list(candidates[s1_id])[:500])

# Ensure all S1 have entry (singletons)
for eid in s1_ids:
    if eid not in candidates:
        candidates[eid] = set()

with open(PROCESSED / 'candidates.pkl', 'wb') as f:
    pickle.dump(dict(candidates), f)
print(f'Candidates saved. Total pairs: {sum(len(v) for v in candidates.values()):,}')

In [ ]:
# ─── CELL 7: Measure blocking recall on training data ────────────────────────
total, covered, missed = 0, 0, []
for s1_id, true_ids in gt_dict.items():
    if not true_ids:
        continue
    cands = candidates.get(s1_id, set())
    for tid in true_ids:
        total += 1
        if tid in cands:
            covered += 1
        else:
            missed.append((s1_id, tid))

recall = covered / total if total > 0 else 1.0
print(f'BLOCKING RECALL: {recall:.4f}  ({covered:,}/{total:,} pairs covered)')
print(f'Missed pairs: {len(missed):,}')
if missed:
    print('Sample missed pairs:')
    for s1_id, tid in missed[:5]:
        s1_row = s1p[s1p['entity_id']==s1_id].iloc[0]
        src = s2p if tid.startswith('S2-') else s3p
        t_row = src[src['entity_id']==tid]
        if not t_row.empty:
            t_row = t_row.iloc[0]
            print(f'  S1: {s1_row["name_clean"][:50]} | True: {t_row["name_clean"][:50]}')

In [ ]:
# ─── CELL 8: Feature Engineering ─────────────────────────────────────────────
from rapidfuzz import fuzz as rffuzz, distance as rfdist

def safe_jac(a, b):
    a, b = set(str(a).split()), set(str(b).split())
    if not a and not b: return 0.0
    return len(a & b) / len(a | b)

def char3_jac(a, b):
    def c3(t):
        t = f' {t} '
        return {t[i:i+3] for i in range(len(t)-2)}
    s1, s2 = c3(str(a)), c3(str(b))
    if not s1 and not s2: return 0.0
    return len(s1 & s2) / len(s1 | s2)

def safe_div(a, b):
    return a / b if b > 0 else 0.0

def compute_features(row_s1, row_pool):
    n1, n2 = str(row_s1.get('name_clean','')), str(row_pool.get('name_clean',''))
    a1, a2 = str(row_s1.get('addr_clean','')), str(row_pool.get('addr_clean',''))
    p1, p2 = str(row_s1.get('postal_code','')).strip(), str(row_pool.get('postal_code','')).strip()

    t1, t2 = set(n1.split()), set(n2.split())
    at1, at2 = set(a1.split()), set(a2.split())

    d = {}
    # Name features
    d['name_exact']          = int(n1 == n2 and n1 != '')
    d['name_jac_word']       = safe_jac(n1, n2)
    d['name_jac_char3']      = char3_jac(n1, n2)
    d['name_lev']            = 1 - rfdist.Levenshtein.normalized_distance(n1, n2) if n1 and n2 else -1
    d['name_jaro']           = rffuzz.WRatio(n1, n2)/100 if n1 and n2 else -1
    d['name_ratio']          = rffuzz.ratio(n1, n2)/100 if n1 and n2 else -1
    d['name_partial']        = rffuzz.partial_ratio(n1, n2)/100 if n1 and n2 else -1
    d['name_tsort']          = rffuzz.token_sort_ratio(n1, n2)/100 if n1 and n2 else -1
    d['name_tset']           = rffuzz.token_set_ratio(n1, n2)/100 if n1 and n2 else -1
    d['name_common_toks']    = len(t1 & t2)
    d['name_tok_ratio']      = safe_div(len(t1 & t2), max(len(t1), len(t2), 1))
    d['name_len_ratio']      = safe_div(min(len(n1),len(n2)), max(len(n1),len(n2),1))
    d['name_first_tok']      = int(n1.split()[0]==n2.split()[0] if n1.split() and n2.split() else False)
    d['name_is_subset']      = int(t1.issubset(t2) or t2.issubset(t1))
    # Address features
    d['addr_exact']          = int(a1==a2 and a1!='')
    d['addr_jac_word']       = safe_jac(a1, a2)
    d['addr_jac_char3']      = char3_jac(a1, a2)
    d['addr_lev']            = 1 - rfdist.Levenshtein.normalized_distance(a1, a2) if a1 and a2 else -1
    d['addr_tsort']          = rffuzz.token_sort_ratio(a1, a2)/100 if a1 and a2 else -1
    d['addr_tset']           = rffuzz.token_set_ratio(a1, a2)/100 if a1 and a2 else -1
    d['addr_common_toks']    = len(at1 & at2)
    d['addr_tok_ratio']      = safe_div(len(at1 & at2), max(len(at1), len(at2), 1))
    d['addr_len_ratio']      = safe_div(min(len(a1),len(a2)), max(len(a1),len(a2),1))
    d['postal_exact']        = int(bool(p1) and p1==p2)
    d['postal_pfx3']         = int(bool(p1) and bool(p2) and p1[:3]==p2[:3])
    d['both_postal']         = int(bool(p1) and bool(p2))
    # Meta + interaction
    c1 = str(row_s1.get('country_norm','')).lower()
    c2 = str(row_pool.get('country_norm','')).lower()
    d['country_eq']          = int(c1==c2 and c1!='')
    d['is_s2']               = int(str(row_pool.get('entity_id','')).startswith('S2-'))
    d['high_name_postal']    = int(d['name_jac_word']>0.7 and d['postal_exact']==1)
    d['high_name_addr']      = int(d['name_jac_word']>0.6 and d['addr_jac_word']>0.6)
    d['name_subset_postal']  = int(d['name_is_subset']==1 and d['postal_exact']==1)
    return d

print('Building feature matrix from candidate pairs ...')
s1_idx   = s1p.set_index('entity_id')
pool_idx = pool_df.set_index('entity_id')

rows = []
total_pairs = sum(len(v) for v in candidates.values())
done = 0

for s1_id, cand_ids in candidates.items():
    if s1_id not in s1_idx.index:
        continue
    r1 = s1_idx.loc[s1_id]
    true_ids = gt_dict.get(s1_id, set())
    for cid in cand_ids:
        if cid not in pool_idx.index:
            continue
        r2   = pool_idx.loc[cid]
        feat = compute_features(r1, r2)
        feat['source1_entity_id']   = s1_id
        feat['candidate_entity_id'] = cid
        feat['label']               = int(cid in true_ids)
        rows.append(feat)
        done += 1
        if done % 500000 == 0:
            print(f'  {done:,}/{total_pairs:,} pairs processed')

feat_df = pd.DataFrame(rows)
feat_df.to_parquet(PROCESSED / 'train_features.parquet', index=False)
print(f'Feature matrix: {feat_df.shape}  Positives: {feat_df["label"].sum():,}')

In [ ]:
# ─── CELL 9: Train/Val split + LightGBM training ─────────────────────────────
import lightgbm as lgb
import pickle

SEED = 42
np.random.seed(SEED)

# Entity-level split
all_s1 = s1p['entity_id'].tolist()
np.random.shuffle(all_s1)
n = len(all_s1)
train_ids = set(all_s1[:int(n*0.70)])
val_ids   = set(all_s1[int(n*0.70):int(n*0.85)])

FEAT_COLS = [c for c in feat_df.columns
             if c not in {'source1_entity_id','candidate_entity_id','label'}]

train_df = feat_df[feat_df['source1_entity_id'].isin(train_ids)]
val_df   = feat_df[feat_df['source1_entity_id'].isin(val_ids)]

# Hard negatives (neg_ratio=5)
pos = train_df[train_df['label']==1]
neg = train_df[train_df['label']==0].sample(n=min(len(pos)*5, len(train_df[train_df['label']==0])), random_state=SEED)
train_set = pd.concat([pos, neg]).sample(frac=1, random_state=SEED)

X_tr, y_tr = train_set[FEAT_COLS].values, train_set['label'].values
X_vl, y_vl = val_df[FEAT_COLS].values,   val_df['label'].values

print(f'Training: {len(train_set):,}  (pos={int(y_tr.sum()):,})  Val: {len(val_df):,}')

model = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.03, num_leaves=127,
    min_child_samples=30, feature_fraction=0.8, bagging_fraction=0.8,
    bagging_freq=5, class_weight='balanced', n_jobs=-1, random_state=SEED
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_vl, y_vl)],
    eval_metric='binary_logloss',
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(100)]
)
print(f'Best iteration: {model.best_iteration_}')

with open(MODELS_DIR / 'lgbm_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'feat_cols': FEAT_COLS}, f)
print('Model saved.')

In [ ]:
# ─── CELL 10: Threshold search on validation set ──────────────────────────────
val_df = val_df.copy()
val_df['score'] = model.predict_proba(X_vl)[:,1]
val_gt = {s: gt_dict[s] for s in val_ids if s in gt_dict}

def entity_f05(pred, true):
    p, t = set(pred), set(true)
    if not p and not t: return 1.0
    tp = len(p & t)
    prec = tp/len(p) if p else 0.0
    rec  = tp/len(t) if t else 0.0
    if prec == 0 or rec == 0: return 0.0
    return (1.25*prec*rec)/(0.25*prec+rec)

best_t, best_f05 = 0.5, 0.0
for t in np.arange(0.05, 0.95, 0.01):
    preds = {s: set() for s in val_gt}
    above = val_df[val_df['score'] >= t]
    for _, row in above.iterrows():
        if row['source1_entity_id'] in preds:
            preds[row['source1_entity_id']].add(row['candidate_entity_id'])
    scores = [entity_f05(preds.get(s,set()), gt_dict.get(s,set())) for s in val_gt]
    mf05 = np.mean(scores)
    if mf05 > best_f05:
        best_f05, best_t = mf05, t

print(f'Best threshold: {best_t:.4f}  |  Val macro F0.5: {best_f05:.4f}')

In [ ]:
# ─── CELL 11: Preprocess and block TEST set ───────────────────────────────────
print('Loading test files ...')
ts1 = pd.read_csv(TEST_DIR/'test_source1.tsv', sep='\t', dtype=str).fillna('')
ts2 = pd.read_csv(TEST_DIR/'test_source2.tsv', sep='\t', dtype=str).fillna('')
ts3 = pd.read_csv(TEST_DIR/'test_source3.tsv', sep='\t', dtype=str).fillna('')
print(f'Test: S1={len(ts1):,}  S2={len(ts2):,}  S3={len(ts3):,}')

ts1p = preprocess_df(ts1)
ts2p = preprocess_df(ts2)
ts3p = preprocess_df(ts3)
tpool = pd.concat([ts2p, ts3p], ignore_index=True)
tpool_ids = tpool['entity_id'].tolist()
tpool_arr = np.array(tpool_ids)
valid_pool_ids = set(tpool_ids)

print('Blocking test set ...')
test_cands = defaultdict(set)

# Pass A: exact name
tname_idx = defaultdict(list)
for _, row in tpool.iterrows():
    if row['name_clean']:
        tname_idx[row['name_clean']].append(row['entity_id'])
for _, row in ts1p.iterrows():
    if row['name_clean'] in tname_idx:
        test_cands[row['entity_id']].update(tname_idx[row['name_clean']])

# Pass B: char TF-IDF (reuse char_vec trained on train pool, re-index on test pool)
tidx_mat  = char_vec.transform(tpool['name_clean'].fillna('').tolist())
for start in range(0, len(ts1p), BATCH):
    end   = min(start+BATCH, len(ts1p))
    batch = ts1p.iloc[start:end]
    q_mat = char_vec.transform(batch['name_clean'].fillna('').tolist())
    sims  = (q_mat @ tidx_mat.T).toarray()
    for i, (_, row) in enumerate(batch.iterrows()):
        top_idx = np.argpartition(sims[i], -min(TOP_K,len(sims[i])))[-TOP_K:]
        top_idx = top_idx[np.argsort(sims[i][top_idx])[::-1]]
        good    = top_idx[sims[i][top_idx]>0]
        test_cands[row['entity_id']].update(tpool_arr[good].tolist())
    if (start//BATCH) % 50 == 0:
        print(f'  test char TF-IDF: {end:,}/{len(ts1p):,}')

# Pass C: token TF-IDF
tidx_mat2 = tok_vec.transform(tpool['name_clean'].fillna('').tolist())
for start in range(0, len(ts1p), BATCH):
    end   = min(start+BATCH, len(ts1p))
    batch = ts1p.iloc[start:end]
    q_mat = tok_vec.transform(batch['name_clean'].fillna('').tolist())
    sims  = (q_mat @ tidx_mat2.T).toarray()
    for i, (_, row) in enumerate(batch.iterrows()):
        top_idx = np.argpartition(sims[i], -min(TOP_K,len(sims[i])))[-TOP_K:]
        top_idx = top_idx[np.argsort(sims[i][top_idx])[::-1]]
        good    = top_idx[sims[i][top_idx]>0]
        test_cands[row['entity_id']].update(tpool_arr[good].tolist())
    if (start//BATCH) % 50 == 0:
        print(f'  test token TF-IDF: {end:,}/{len(ts1p):,}')

# Pass D: postal
tpost_idx = defaultdict(list)
for _, row in tpool.iterrows():
    pc = row['postal_code'].strip()
    if pc:
        key = f"{row['country_norm']}|{pc}"
        tpost_idx[key].append(row['entity_id'])
for _, row in ts1p.iterrows():
    pc = row['postal_code'].strip()
    if pc:
        key = f"{row['country_norm']}|{pc}"
        if key in tpost_idx and len(tpost_idx[key]) <= 500:
            test_cands[row['entity_id']].update(tpost_idx[key])

# Ensure all S1 entities present
for eid in ts1p['entity_id'].tolist():
    if eid not in test_cands:
        test_cands[eid] = set()

# Cap
for eid in list(test_cands.keys()):
    if len(test_cands[eid]) > 500:
        test_cands[eid] = set(list(test_cands[eid])[:500])

print(f'Test candidates: {sum(len(v) for v in test_cands.values()):,} pairs')

In [ ]:
# ─── CELL 12: Feature engineering for test candidates ─────────────────────────
ts1_idx   = ts1p.set_index('entity_id')
tpool_idx = tpool.set_index('entity_id')

test_rows = []
done = 0
total_test = sum(len(v) for v in test_cands.values())

for s1_id, cand_ids in test_cands.items():
    if s1_id not in ts1_idx.index:
        continue
    r1 = ts1_idx.loc[s1_id]
    for cid in cand_ids:
        if cid not in tpool_idx.index:
            continue
        r2   = tpool_idx.loc[cid]
        feat = compute_features(r1, r2)
        feat['source1_entity_id']   = s1_id
        feat['candidate_entity_id'] = cid
        test_rows.append(feat)
        done += 1
        if done % 500000 == 0:
            print(f'  {done:,}/{total_test:,}')

test_feat_df = pd.DataFrame(test_rows)
print(f'Test feature matrix: {test_feat_df.shape}')

In [ ]:
# ─── CELL 13: Generate predictions + write output files ───────────────────────
test_feat_df['score'] = model.predict_proba(test_feat_df[FEAT_COLS].values)[:,1]

# Apply threshold
predictions = {eid: set() for eid in ts1p['entity_id'].tolist()}
above = test_feat_df[test_feat_df['score'] >= best_t]
for _, row in above.iterrows():
    s1_id = row['source1_entity_id']
    cid   = row['candidate_entity_id']
    if cid in valid_pool_ids and not cid.startswith('S1-'):
        predictions[s1_id].add(cid)

n_match = sum(1 for v in predictions.values() if v)
n_sing  = sum(1 for v in predictions.values() if not v)
print(f'Entities with matches: {n_match:,}  Singletons: {n_sing:,}')

# Write matching_results.tsv
match_rows = []
for s1_id, matched in sorted(predictions.items()):
    match_rows.append({'source1_entity_id': s1_id,
                       'matched_entity_ids': ','.join(sorted(matched)) if matched else ''})
match_df = pd.DataFrame(match_rows)
match_df.to_csv(OUTPUT_DIR/'matching_results.tsv', sep='\t', index=False)

# Write candidate_pairs.tsv
cand_rows = []
for s1_id, cands in sorted(test_cands.items()):
    cand_rows.append({'source1_entity_id': s1_id,
                      'candidate_entity_ids': ','.join(sorted(cands)) if cands else ''})
cand_df = pd.DataFrame(cand_rows)
cand_df.to_csv(OUTPUT_DIR/'candidate_pairs.tsv', sep='\t', index=False)

print(f'Output files written to {OUTPUT_DIR}')
print(f'Rows in matching_results: {len(match_df):,}')
print(f'Rows in candidate_pairs:  {len(cand_df):,}')

In [ ]:
# ─── CELL 14: Quick self-validation checks ────────────────────────────────────
test_s1_ids = set(ts1p['entity_id'].tolist())
submission_ids = set(match_df['source1_entity_id'].tolist())

print('VALIDATION CHECKS:')
print(f'  All test S1 present: {test_s1_ids == submission_ids}')
print(f'  No duplicate S1 rows: {not match_df["source1_entity_id"].duplicated().any()}')

# Check no S1 IDs in match list
self_matches = 0
for _, row in match_df.iterrows():
    if row['matched_entity_ids']:
        for mid in row['matched_entity_ids'].split(','):
            if mid.startswith('S1-'):
                self_matches += 1
print(f'  No self-matches (S1-*): {self_matches == 0}')

print(f'\nmatch_df shape: {match_df.shape}')
print(f'cand_df shape:  {cand_df.shape}')
print('\nSample output:')
print(match_df[match_df['matched_entity_ids']!=''].head(5).to_string())

In [ ]:
# ─── CELL 15: Download submission files ──────────────────────────────────────
from google.colab import files
files.download(str(OUTPUT_DIR / 'matching_results.tsv'))
files.download(str(OUTPUT_DIR / 'candidate_pairs.tsv'))
print('Files downloaded. Upload matching_results.tsv to the portal!')